In [1]:
import torch
from collections import defaultdict
from typing import Dict, List, Optional, Tuple



def compute_category_similarity_matrix(
    category_to_centroid: Dict[str, torch.Tensor]
) -> Dict[str, Dict[str, float]]:
    """
    args:
    - category_to_centroid: カテゴリ名 -> セントロイドベクトル

    カテゴリ同士の cosine similarity の辞書を返す。
    sim[a][b] = cosine(category_a, category_b)
    """
    categories = list(category_to_centroid.keys())
    sim = defaultdict(dict)

    for cat_a in categories:
        vec_a = category_to_centroid[cat_a]
        for cat_b in categories:
            vec_b = category_to_centroid[cat_b]
            score = torch.dot(vec_a, vec_b).item()
            sim[cat_a][cat_b] = score

    return dict(sim)


def classify_other_categories(
    similarity_matrix: Dict[str, Dict[str, float]],
    top_k_near: int = 3,
    top_k_far: int = 3,
    allowed_categories: Optional[List[str]] = None,
) -> Dict[str, Dict[str, List[Tuple[str, float]]]]:
    """
    args:
    - similarity_matrix: compute_category_similarity_matrix の出力
    - top_k_near: 近い他カテゴリの数
    - top_k_far: 遠い他カテゴリの数
    - allowed_categories: 分類対象とするカテゴリのリスト

    各 own_category に対して
    - near: 近い他カテゴリ
    - far: 遠い他カテゴリ
    - middle: 中間カテゴリ
    を返す。
    """
    result = {}
    allowed_set = set(allowed_categories) if allowed_categories is not None else None

    for own_category, row in similarity_matrix.items():
        candidates = []
        for other_category, score in row.items():
            if other_category == own_category:
                continue
            if allowed_set is not None and other_category not in allowed_set:
                continue
            candidates.append((other_category, score))

        # 類似度が高い順
        candidates_sorted = sorted(candidates, key=lambda x: x[1], reverse=True)

        near = candidates_sorted[:top_k_near]
        far = candidates_sorted[-top_k_far:] if top_k_far > 0 else []

        near_names = {x[0] for x in near}
        far_names = {x[0] for x in far}
        middle = [x for x in candidates_sorted if x[0] not in near_names and x[0] not in far_names]

        result[own_category] = {
            "near": near,
            "middle": middle,
            "far": far,
        }

    return result

In [2]:
import json
import os
import sys

# プロジェクトのutils追加
project_root = os.path.join(os.getcwd(), "..")
sys.path.append(project_root)

from utils.wikipedia_api_utils import load_wikisummary
from utils.handle_text_utils import get_first_few_sentences, repeat_text, delete_non_English_characters


wiki_pages_dir = os.path.join(project_root, "data", "wiki_pages")
config_path = "../config/target_concepts_mini_13.json"


with open(config_path, "r") as f:
    config = json.load(f)


propnoun_to_repeatwikisummary = {}
for cat, propnouns in config.items():
    print(f"{cat}: {propnouns}")

    for propnoun in propnouns:
        if propnoun.strip() == "":
            continue
        # ** このterm(prop noun)を説明する wiki page の summary を取得し、前処理を行う **
        summary = load_wikisummary(propnoun, wiki_pages_dir)

        # 短すぎor長すぎるsummaryがあるため、最初の数文だけをsummaryとして使用する. (30~300単語に収まるように調整) 30単語未満のsummaryは、十分な情報が得られない可能性があるため、初期化vecの計算に使用しない. 
        min_words, max_words = 30, 300 # 30->50に変更すると、そこまで長いsummaryが少ないようで、init vecが0vecとなりlossがNanになってしまった。minは30でキープする
        summary = get_first_few_sentences(summary, min_words, max_words)
        if summary is None:
            print(f"'{propnoun}' のWikipedia summaryは、{min_words} ~ {max_words}単語の範囲内に収まらないため、スキップします。") # 最初の100文字だけ表示
            continue

        # 英語数字記号以外の文字を削除
        summary = delete_non_English_characters(summary)

        # *** 初期vecを、固有名詞毎のwikiのsummary文を2回入力して、2回目の文内token位置の隠れ状態から作る場合: https://openreview.net/forum?id=Ahlrf2HGJR の手法 ***
        summary = repeat_text(summary, 2)

        propnoun_to_repeatwikisummary[propnoun] = summary

propnoun_to_repeatwikisummary

Algorithm: []
Archive: []
Band: ["Ego-Wrappin'", 'Pomplamoose', 'Yello', 'Doina and Ion Aldea Teodorovici', 'Adam & Eve', 'ABBA']
Election: []
Food: []
Historical region: []
Intercommunality: []
Legal Case: []
Medicine: []
Noble family: ['House of Tudor', 'Carolingian dynasty', 'Kassites', 'House of Hohenstaufen', 'Ayyubid dynasty']
Painting: ['Azuchi Screens', 'Our Lady of the Sign', 'Lehighton', 'Murals of Chapel Hill', 'Wagner Murals']
skip.  word count: 22 < 30, char_count: 133, The Wagner Murals are the name for over 70 mural fragments illegally removed from the Pre-Columbian site of Teotihuacán in the 1960s....
'Wagner Murals' のWikipedia summaryは、30 ~ 300単語の範囲内に収まらないため、スキップします。
Port: []
Site of Special Scientific Interest: []
Wikimedia template: []
Windmill: []
academic subject: []
activity: []
agent: []
aircraft: []
airline: []
airport: []
album: []
anatomical structure: []
animal: []
archipelago: []
artery: []
artwork: []
asteroid: []
award: []
board game: ['1942', 'Dameo', 'Hi

{"Ego-Wrappin'": 'Ego-Wrappin\' (stylised as EGO-WRAPPIN\') is a Japanese jazz and rock musical duo, composed of vocalist Yoshie Nakano and guitarist Masaki Mori. The group formed in Osaka in 1996, releasing their debut album Blue Speaker in 1998. The band gained national recognition with their cabaret and kaykyoku inspired song "Midnight Dejavu (Shikisai no Blues)" (2000). Ego-Wrappin\' (stylised as EGO-WRAPPIN\') is a Japanese jazz and rock musical duo, composed of vocalist Yoshie Nakano and guitarist Masaki Mori. The group formed in Osaka in 1996, releasing their debut album Blue Speaker in 1998. The band gained national recognition with their cabaret and kaykyoku inspired song "Midnight Dejavu (Shikisai no Blues)" (2000).',
 'Pomplamoose': 'Pomplamoose () is an American musical husband-and-wife duo composed of multi-instrumentalist Jack Conte and singer-songwriter and bassist Nataly Dawn. Formed in 2008, the duo sold about 100,000 songs online in 2009. They are known for their vira